In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

/kaggle/input/nyc-property-sales/nyc-rolling-sales.csv
/kaggle/input/india-trade-data/2018-2010_export.csv
/kaggle/input/india-trade-data/2018-2010_import.csv


![](https://docs.dask.org/en/latest/_images/dask_icon.svg)

# Introduction to Dask

- If data is more than RAM size, certainly a Data Scientist will look for another tool and first tool which might come in sight is Dask. In Dask the Dask DataFrame is no doubtly going to be a huge support to analyse huge amount of data. A Dask DataFrame, is made of many small Pandas DataFrame.
- Can run in parallel system like cluster or on single machine.
- If Data is larger than RAM. No Problem some part will be in memory some part on disc but computation go on single machine. 

# Installation 

- On Kaggle machines Dask is allready installed. 
- But if you want to install Dask on any machine then it can be installed using conda and pip both
- Installing Dask using conda using following line of code

conda install dask

- If you have installed Anaconda then Dask will be installed with it. Further if required it can be upgraded. 

- Dask can be installed using pip too.

pip install dask  : But it will only install core Dask Part

- In order to install complete Dask following code can be used

pip install "dask[complete]"
conda install "dask[complete]"

# About DataSet used for this kernel

Dataset i have used from kaggle itself. This Dataset about importing stuffs to India and Exporting Stuffs from India. Following Information i have just copy pasted from Data Official site on kaggle 

---------------------------------------------------

- Context
India is one of the fastest developing nations of the world and trade between nations is the major component of any developing nation. This dataset includes the trade data for India for commodities in the HS2 basket.

For more, visit GitHub

- Content
The dataset consists of trade values for export and import of commodities in million US$. The dataset is tidy and each row consists of a single observation.

- Acknowledgements
The data is scraped using Selenium Webdriver from the Department of Commerce, Government of India.

Data set has been collect and uploaded to Kaggle by  Lakshya Agarwal  
https://www.kaggle.com/lakshyaag



# More about Dataset 

taken from kernel 
https://www.kaggle.com/shubhamsinghgharsele/analysis-on-indian-import-export


In both the File we have 5 columns each.

HSCode - HS stands for Harmonized System. It was developed by the WCO (World Customs Organization) as a multipurpose international product nomenclature that describes the type of good that is shipped HS Code Structure

The HS code can be described as follows:

It is a six-digit identification code. It has 5000 commodity groups. Those groups have 99 chapters. Those chapters have 21 sections. It’s arranged in a legal and logical structure. Well-defined rules support it to realize uniform classification worldwide.

the HSCode in column is 99 chapters

Reference HSCode List

Commodity - the column contain chapter wise commodity category. In each commodity Category there are various commodities.

A commodity is an economic good or service that has full or substantial fungibility: that is, the market treats instances of the good as equivalent or nearly so with no regard to who produced them.

Reference

Value - values for export and import of commodities in million US $.
Country - Country Imported From/ Exported To
Year - Year in which comodities where Imported/Exported which is in between 2010 to 2018.

# Importing Dask 

In [2]:
 import dask.dataframe as dd

# Reading the Data Files

In [3]:
exportDf = dd.read_csv('/kaggle/input/india-trade-data/2018-2010_export.csv')

In [4]:
exportDf.head()

,HSCode,Commodity,value,country,year
0,2,MEAT AND EDIBLE MEAT OFFAL.,0.18,AFGHANISTAN TIS,2018
1,3,"FISH AND CRUSTACEANS, MOLLUSCS AND OTHER AQUAT...",0.00,AFGHANISTAN TIS,2018
2,4,DAIRY PRODUCE; BIRDS' EGGS; NATURAL HONEY; EDI...,12.48,AFGHANISTAN TIS,2018
3,6,LIVE TREES AND OTHER PLANTS; BULBS; ROOTS AND ...,0.00,AFGHANISTAN TIS,2018
4,7,EDIBLE VEGETABLES AND CERTAIN ROOTS AND TUBERS.,1.89,AFGHANISTAN TIS,2018


In [5]:
# Finding information about DataFrame
exportDf.info()

<class 'dask.dataframe.core.DataFrame'>
Columns: 5 entries, HSCode to year
dtypes: object(2), float64(1), int64(2)

In [6]:
# Schema of DataFrame
exportDf.dtypes

HSCode         int64
Commodity     object
value        float64
country       object
year           int64
dtype: object

In [7]:
importDf = dd.read_csv('/kaggle/input/india-trade-data/2018-2010_import.csv')

In [8]:
importDf.head()

,HSCode,Commodity,value,country,year
0,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.00,AFGHANISTAN TIS,2018
1,7,EDIBLE VEGETABLES AND CERTAIN ROOTS AND TUBERS.,12.38,AFGHANISTAN TIS,2018
2,8,EDIBLE FRUIT AND NUTS; PEEL OR CITRUS FRUIT OR...,268.60,AFGHANISTAN TIS,2018
3,9,"COFFEE, TEA, MATE AND SPICES.",35.48,AFGHANISTAN TIS,2018
4,11,PRODUCTS OF THE MILLING INDUSTRY; MALT; STARCH...,NaN,AFGHANISTAN TIS,2018


In [9]:

set(importDf.HSCode) == set(exportDf.HSCode)

True

# Function compute perform the computation.

In [10]:
importDf.value.min().compute()

0.0

# Data Filtering

In [11]:
importDf.describe().compute()

,HSCode,value,year
count,93095.000000,79068.000000,93095.000000
mean,53.849573,63.289855,2014.654740
std,27.567486,666.652363,2.702373
min,1.000000,0.000000,2010.000000
25%,30.000000,0.030000,2012.000000
50%,54.000000,0.380000,2015.000000
75%,78.000000,4.910000,2017.000000
max,99.000000,32781.570000,2018.000000


In [12]:
#Getting number of partition
importDf.npartitions

1

# Data Filtering 
- Getting all the imports where import value is greater than 66

In [13]:
filteredVal =importDf[importDf.value > 66]
filteredVal.head()

,HSCode,Commodity,value,country,year
2,8,EDIBLE FRUIT AND NUTS; PEEL OR CITRUS FRUIT OR...,268.60,AFGHANISTAN TIS,2018
6,13,"LAC; GUMS, RESINS AND OTHER VEGETABLE SAPS AND...",108.78,AFGHANISTAN TIS,2018
55,27,"MINERAL FUELS, MINERAL OILS AND PRODUCTS OF TH...",1559.37,ALGERIA,2018
59,31,FERTILISERS.,83.02,ALGERIA,2018
95,27,"MINERAL FUELS, MINERAL OILS AND PRODUCTS OF TH...",4012.00,ANGOLA,2018


In [14]:
filteredVal =importDf[(importDf.value > 66) & (importDf.value < 100)]
filteredVal.head()

,HSCode,Commodity,value,country,year
59,31,FERTILISERS.,83.02,ALGERIA,2018
241,32,TANNING OR DYEING EXTRACTS; TANNINS AND THEIR ...,95.30,AUSTRALIA,2018
288,84,"NUCLEAR REACTORS, BOILERS, MACHINERY AND MECHA...",87.76,AUSTRALIA,2018
379,85,ELECTRICAL MACHINERY AND EQUIPMENT AND PARTS T...,88.68,AUSTRIA,2018
445,26,"ORES, SLAG AND ASH.",96.51,BAHARAIN IS,2018


# Data Aggregation

- Mean value grouped by  HSCode

In [15]:
meanVal = importDf.groupby("HSCode").mean()
meanVal.compute()

,value,year
HSCode,,
1,0.270967,2014.354237
2,0.231014,2014.469072
3,2.020248,2014.810811
4,1.859447,2014.736944
5,1.046796,2014.469667
...,...,...
95,7.427526,2014.600219
96,5.827932,2014.546112
97,2.188893,2014.929110


In [16]:
# Data joining 
joinedData = importDf.merge(exportDf, on = "HSCode",how="inner")
joinedData.compute().head()

,HSCode,Commodity_x,value_x,country_x,year_x,Commodity_y,value_y,country_y,year_y
0,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.0,AFGHANISTAN TIS,2018,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.32,ALBANIA,2018
1,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.0,AFGHANISTAN TIS,2018,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.00,ALGERIA,2018
2,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.0,AFGHANISTAN TIS,2018,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.06,AUSTRALIA,2018
3,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.0,AFGHANISTAN TIS,2018,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.02,AUSTRIA,2018
4,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.0,AFGHANISTAN TIS,2018,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.01,BAHARAIN IS,2018


# Comparison of performance between pandas and dask DataFrame

In [17]:
# Reading a File using Pandas
%timeit pd.read_csv("/kaggle/input/india-trade-data/2018-2010_import.csv")

106 ms ± 4.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
%timeit dd.read_csv("/kaggle/input/india-trade-data/2018-2010_import.csv")

13 ms ± 344 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## For file reading Dask is much faster than Pandas

### Comparison of code efficiency for Data Filtering

In [19]:
importDfPd = pd.read_csv('/kaggle/input/india-trade-data/2018-2010_import.csv')
importDfDd = dd.read_csv('/kaggle/input/india-trade-data/2018-2010_import.csv')

In [20]:
# Filtering using Dask
%timeit val = importDfDd[importDfDd.value >66].compute()

117 ms ± 1.98 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [21]:
# Filtering using Pandas
%timeit val = importDfPd[importDfPd.value >66]

1.67 ms ± 25.6 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


### We can observe that time is very large for Dask. Can we decrease it? Let us increase number of Partitions.

In [22]:
importDfDd.head()

,HSCode,Commodity,value,country,year
0,5,"PRODUCTS OF ANIMAL ORIGIN, NOT ELSEWHERE SPECI...",0.00,AFGHANISTAN TIS,2018
1,7,EDIBLE VEGETABLES AND CERTAIN ROOTS AND TUBERS.,12.38,AFGHANISTAN TIS,2018
2,8,EDIBLE FRUIT AND NUTS; PEEL OR CITRUS FRUIT OR...,268.60,AFGHANISTAN TIS,2018
3,9,"COFFEE, TEA, MATE AND SPICES.",35.48,AFGHANISTAN TIS,2018
4,11,PRODUCTS OF THE MILLING INDUSTRY; MALT; STARCH...,NaN,AFGHANISTAN TIS,2018


In [23]:
importDfDd1 = importDfDd.repartition(npartitions = 100)

In [24]:
importDfDd.npartitions

1

In [25]:
# Filtering using Dask
%timeit val = importDfDd1[importDfDd1.value >66].compute()

454 ms ± 6.49 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
importDfDd1 = importDfDd.repartition(npartitions = 1)
%timeit val = importDfDd1[importDfDd1.value >66].compute()

118 ms ± 1.77 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
